# 4 — Weighted Domain Accuracy (WDA) evaluation

Evaluate the point-detection model's predictions on the **GWHD 2021** test set using the
**Weighted Domain Accuracy (WDA)** metric from David et al. 2021, *Global Wheat Head Detection
(GWHD) Dataset* ([Plant Phenomics, 10.34133/2021/9846158](https://doi.org/10.34133/2021/9846158)).

## The metric

The accuracy over image $i$ belonging to domain $d$ is

$$\mathrm{AI}_d(i) = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN} + \mathrm{FP}},$$

where $\mathrm{TP}$, $\mathrm{FN}$, $\mathrm{FP}$ are the numbers of true positives, false
negatives and false positives in image $i$. The **weighted domain accuracy** is the mean of the
per-domain mean accuracies:

$$\mathrm{WDA} = \frac{1}{D}\sum_{d=1}^{D}\; \frac{1}{n_d}\sum_{i=1}^{n_d} \mathrm{AI}_d(i),$$

where $D$ is the number of domains (sub-datasets) and $n_d$ is the number of images in domain $d$.
Note that WDA weights every **domain** equally, regardless of how many images it contains.

## Matching TP / FP / FN

The original GWHD challenge matches predicted **boxes** to ground-truth **boxes** at $\mathrm{IoU}\ge0.5$
with one-to-one greedy assignment. Our model outputs **points**, so we adapt the matching to a
parameter-free **point-in-box** rule against the ground-truth boxes:

* Predictions are considered in **descending confidence** order.
* A predicted point is a **TP** if it falls inside a still-unmatched GT box; that box is then consumed
  (one-to-one). If the point lands inside several unmatched boxes, the one whose centre is nearest is used.
* A predicted point matching no free box is a **FP**.
* Every GT box left unmatched is a **FN**.
* Empty image with no predictions -> $\mathrm{AI}=1$ (the $0/0$ case is defined as a perfect score).

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Paths -------------------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PRED_XML = ROOT / "data/inference/runs/20260819_104808_best/gwhd_2021_reformat_test/predictions_cvat.xml"
GT_CSV   = ROOT / "data/gwhd_2021/competition_test.csv"

assert PRED_XML.exists(), PRED_XML
assert GT_CSV.exists(), GT_CSV
print("Predictions:", PRED_XML)
print("Ground truth:", GT_CSV)

In [ ]:
# --- Parse the CVAT point predictions ---------------------------------------
# Returns {image_name: np.array of shape (N, 3) -> (x, y, confidence)}.
# CVAT stores confidence as an integer 0-100 in a child <attribute name="Confidence">.

def parse_pred_points(xml_path):
    root = ET.parse(xml_path).getroot()
    preds = {}
    for img in root.findall("image"):
        name = img.get("name")
        pts = []
        for p in img.findall("points"):
            x, y = map(float, p.get("points").split(","))
            conf_el = p.find("attribute")
            conf = float(conf_el.text) if conf_el is not None else 100.0
            pts.append((x, y, conf))
        preds[name] = np.array(pts, dtype=float).reshape(-1, 3)
    return preds

predictions = parse_pred_points(PRED_XML)
n_pts = sum(len(v) for v in predictions.values())
all_conf = np.concatenate([v[:, 2] for v in predictions.values() if len(v)])
print(f"{len(predictions)} images, {n_pts} predicted points")
print(f"confidence range: {all_conf.min():.0f} - {all_conf.max():.0f}")

In [ ]:
# --- Parse the ground-truth boxes + domains ---------------------------------
# competition_test.csv columns: image_name, BoxesString, domain
# BoxesString is ";"-separated "x1 y1 x2 y2", or the literal "no_box" for empty images.
# One image_name appears twice in the CSV -> we merge the box lists.

def parse_boxes(s):
    s = s.strip()
    if not s or s.lower() == "no_box":
        return np.zeros((0, 4), dtype=float)
    boxes = [list(map(float, b.split())) for b in s.split(";") if b.strip()]
    return np.array(boxes, dtype=float).reshape(-1, 4)  # (M, 4) x1,y1,x2,y2

gt_df = pd.read_csv(GT_CSV)
gt_boxes, gt_domain = {}, {}
for _, r in gt_df.iterrows():
    name = r["image_name"]
    boxes = parse_boxes(str(r["BoxesString"]))
    if name in gt_boxes:  # duplicate row -> merge boxes
        gt_boxes[name] = np.vstack([gt_boxes[name], boxes])
    else:
        gt_boxes[name] = boxes
        gt_domain[name] = r["domain"]

domains = sorted(set(gt_domain.values()))
print(f"{len(gt_boxes)} GT images across {len(domains)} domains")
print(f"total GT boxes: {sum(len(b) for b in gt_boxes.values())}")

# Sanity: every prediction image must exist in the GT.
missing = set(predictions) - set(gt_boxes)
assert not missing, f"{len(missing)} predicted images not in GT: {list(missing)[:3]}"

In [ ]:
# --- Point-in-box matching -> TP / FP / FN for one image --------------------

def match_image(points_xyc, boxes, conf_thr=0.0):
    """Greedy one-to-one point-in-box matching, predictions ranked by confidence.

    points_xyc : (N, 3) array of (x, y, confidence)
    boxes      : (M, 4) array of (x1, y1, x2, y2)
    conf_thr   : keep only predictions with confidence >= conf_thr
    Returns (TP, FP, FN).
    """
    pts = points_xyc[points_xyc[:, 2] >= conf_thr] if len(points_xyc) else points_xyc
    M = len(boxes)
    if M == 0:
        return 0, len(pts), 0  # no GT: every kept prediction is a FP
    if len(pts) == 0:
        return 0, 0, M         # no predictions: every GT box is a FN

    order = np.argsort(-pts[:, 2])  # highest confidence first
    box_used = np.zeros(M, dtype=bool)
    box_cx = (boxes[:, 0] + boxes[:, 2]) / 2.0
    box_cy = (boxes[:, 1] + boxes[:, 3]) / 2.0
    tp = fp = 0
    for idx in order:
        x, y = pts[idx, 0], pts[idx, 1]
        inside = (~box_used) & (boxes[:, 0] <= x) & (x <= boxes[:, 2]) & \
                 (boxes[:, 1] <= y) & (y <= boxes[:, 3])
        if inside.any():
            cand = np.where(inside)[0]
            # nearest box centre among the containing, still-free boxes
            j = cand[np.argmin((box_cx[cand] - x) ** 2 + (box_cy[cand] - y) ** 2)]
            box_used[j] = True
            tp += 1
        else:
            fp += 1
    fn = int((~box_used).sum())
    return tp, fp, fn

In [ ]:
# --- WDA computation ---------------------------------------------------------

def image_accuracy(tp, fp, fn):
    denom = tp + fn + fp
    return 1.0 if denom == 0 else tp / denom  # 0/0 (empty & no preds) := perfect

def compute_wda(predictions, gt_boxes, gt_domain, conf_thr=0.0, return_details=False):
    """Return WDA at a given confidence threshold (plus optional per-image table)."""
    per_domain_acc = defaultdict(list)
    rows = []
    for name, boxes in gt_boxes.items():
        pts = predictions.get(name, np.zeros((0, 3)))
        tp, fp, fn = match_image(pts, boxes, conf_thr)
        acc = image_accuracy(tp, fp, fn)
        d = gt_domain[name]
        per_domain_acc[d].append(acc)
        if return_details:
            rows.append((name, d, tp, fp, fn, acc))

    domain_acc = {d: float(np.mean(a)) for d, a in per_domain_acc.items()}
    wda = float(np.mean(list(domain_acc.values())))
    if return_details:
        details = pd.DataFrame(rows, columns=["image", "domain", "TP", "FP", "FN", "AI"])
        return wda, domain_acc, details
    return wda, domain_acc

wda, domain_acc = compute_wda(predictions, gt_boxes, gt_domain, conf_thr=0.0)
print(f"WDA (all predictions, conf >= 0): {wda:.4f}")

In [ ]:
# --- Sweep the confidence threshold -----------------------------------------
# Low-confidence points inflate FP; a threshold trades FP against FN. We report
# the operating point that maximises WDA on the test set.

thresholds = np.arange(all_conf.min(), all_conf.max() + 1)
sweep = [(t, compute_wda(predictions, gt_boxes, gt_domain, conf_thr=t)[0]) for t in thresholds]
sweep = pd.DataFrame(sweep, columns=["conf_thr", "WDA"])

best = sweep.loc[sweep["WDA"].idxmax()]
print(f"Best WDA = {best.WDA:.4f} at confidence threshold {best.conf_thr:.0f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep["conf_thr"], sweep["WDA"], lw=2)
ax.axvline(best.conf_thr, color="crimson", ls="--", lw=1,
           label=f"best thr={best.conf_thr:.0f}, WDA={best.WDA:.4f}")
ax.set_xlabel("confidence threshold")
ax.set_ylabel("WDA")
ax.set_title("WDA vs. confidence threshold (GWHD 2021 test)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Per-domain breakdown at the best threshold -----------------------------

best_thr = float(best.conf_thr)
wda_best, domain_acc_best, details = compute_wda(
    predictions, gt_boxes, gt_domain, conf_thr=best_thr, return_details=True
)

agg = (details.groupby("domain")
       .agg(n_images=("image", "size"), TP=("TP", "sum"), FP=("FP", "sum"),
            FN=("FN", "sum"), mean_AI=("AI", "mean"))
       .reset_index()
       .sort_values("mean_AI", ascending=False))

print(f"WDA @ conf>={best_thr:.0f}: {wda_best:.4f}   (domains equally weighted)\n")
print(agg.to_string(index=False, formatters={"mean_AI": "{:.4f}".format}))

# Micro-accuracy (pooled TP/FP/FN, for reference -- NOT the WDA)
TP, FP, FN = details[["TP", "FP", "FN"]].sum()
print(f"\nPooled micro-accuracy = {TP/(TP+FP+FN):.4f}  (TP={TP}, FP={FP}, FN={FN})")

In [ ]:
# --- Bar chart of per-domain accuracy ---------------------------------------

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(agg["domain"], agg["mean_AI"], color="#4C9A6B")
ax.axhline(wda_best, color="crimson", ls="--", lw=1.2, label=f"WDA = {wda_best:.4f}")
ax.set_ylabel(r"$\mathrm{AI}_d$ (mean image accuracy)")
ax.set_title(f"Per-domain accuracy @ conf>={best_thr:.0f}")
ax.set_ylim(0, 1)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.show()